# 18 — Sintéticos IV: aumento de datos (real + sintético)

¿Entrenar los detectores del benchmark con **real + sintético** mejora su detección de las crisis
**reales** fuera de muestra? Se evalúa solo sobre el real OOS, con el mismo `walk_forward`, las mismas
specs y las mismas métricas ADR-003 que el benchmark; la referencia (proporción 0) son sus paneles
OOS, así que la comparación es pareada día a día. Todo el diseño está pre-registrado (2026-10-15,
antes de ejecutar nada) en [`configs/sinteticos.yaml → aumento`](../configs/sinteticos.yaml), con sus
concreciones y erratas fechadas. Ninguna cifra de este texto está escrita a mano: salen de las celdas.

## 0. Por qué este alcance

- **Generadores: los `apto_aumento` del 16 en las dos pistas** — `gaussiano_regimen`, `var_regimen`,
  `garch_regimen`, `rbig` y `flow_matching`. El criterio lo fijó el 16, no este notebook, y es el mismo
  conjunto en A y B para que las pistas sean comparables. `bootstrap_regimen` (días reales remuestreados)
  no es apto porque memoriza: entra **solo como control** («¿basta con cualquier dato extra?»).
- **Por qué no el alcance completo.** `difusion`, `cvae` y `cgan` son aptos solo en A (en B no pasan el
  condicionamiento): romperían la comparabilidad entre pistas y añadirían unas 19 h de cálculo. Quedan
  como extensión posible, declarada como tal.
- **Detectores: todos los evaluables** (12 en A; 10 en B, donde D02 y D10 usan columnas que los
  generadores no producen). Se descartó recortar A al top-8 del ranking porque dejaba fuera justo a los
  que aprenden un estado de crisis (D04, D08, D09, D11), los que más podrían beneficiarse: el recorte
  sesgaba en contra de encontrar efecto. Un resultado nulo solo es creíble si es amplio.
- **Expectativa declarada: efecto pequeño o nulo.** Los detectores son no supervisados (el aumento solo
  cambia la ley que ven al ajustarse); en el TSTR del 16, entrenar solo con sintético quedó casi siempre
  por debajo del real; y en las reglas de umbral el aumento recalibra umbrales en vez de aprender.
- **Coste.** Unas 400–460 h de CPU, ejecutadas por sesiones de 8 h con 10 procesos
  (`python -m regimenes.sinteticos.aumento ejecutar`, reanudable) y por lotes de prioridad, cada uno con
  resultados completos. Este notebook **solo lee** los CSV consolidados
  (`python -m regimenes.sinteticos.aumento consolidar`) y funciona con resultados parciales: lo que
  falta sale como «falta», nunca como cero.

**Índice.** 1. Configuración y estado · 2. Diseño · 3. H1 (principal) · 4. Mapa detector × generador ·
5. H2 dosis-respuesta · 6. H3, H4 y H5 · 7. Diagnósticos · 8. Regla para el 19 y conclusiones

## 1. Configuración y estado de la ejecución

Pre-registro resumido y cobertura: trabajos hechos y pendientes por lote de prioridad (en el orden en
que se lanzan) y horas de CPU estimadas para lo que falta. La carpeta de resultados es la real salvo que
la variable de entorno `NB18_RESULTADOS` apunte a otra.

In [ ]:
%matplotlib inline
import json
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from regimenes import viz
from regimenes.sinteticos.aumento import analisis as an
from regimenes.sinteticos.aumento import figuras as fg
from regimenes.sinteticos.aumento.anclas import leer_registro, ruta_registro
from regimenes.sinteticos.aumento.config import DIR_RESULTADOS, config_aumento

warnings.filterwarnings('ignore')
pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 110)
viz.use_house_style()

RES = Path(os.environ.get('NB18_RESULTADOS', DIR_RESULTADOS))   # carpeta de resultados (por defecto, la real)
FIGS = RES / 'figuras'
CFG = config_aumento()
PISTAS = CFG['pistas']


def guardar(obj, nombre):
    # tabla -> RES/<nombre>.csv (si no está vacía) ; figura -> FIGS/<nombre>.png (y se muestra)
    if isinstance(obj, pd.DataFrame):
        if not obj.empty:
            obj.to_csv(RES / f'{nombre}.csv', index=False)
        return
    FIGS.mkdir(parents=True, exist_ok=True)
    obj.savefig(FIGS / f'{nombre}.png', dpi=130, bbox_inches='tight')
    plt.show()


pre = {k: CFG[k] for k in ('generadores', 'control', 'excluir', 'proporciones', 'repeticiones', 'trayectoria_largo')}
pre.update({'margen_reconocimiento': CFG['anclas']['margen_reconocimiento'], 'h1': CFG['contrastes']['h1'],
            'mejora_minima': CFG['contrastes']['mejora_minima'], 'n_boot': CFG['contrastes']['n_boot']})
display(pd.Series({k: json.dumps(v, ensure_ascii=False) for k, v in pre.items()}, name='valor').to_frame())
COB = an.cobertura(PISTAS, RES)
display(COB.round(1))
DATOS = an.cargar(PISTAS, RES)
FILAS = an.cargar_filas(PISTAS, RES)
print(f'Trabajos hechos: {int(COB.hechos.sum())} de {int(COB.trabajos.sum())}; '
      f'pendientes ~{COB.horas_cpu_pendientes.sum():.0f} h de CPU. Pistas consolidadas: {list(DATOS) or "ninguna"}')
for p, d in DATOS.items():
    print(f'  {p}: señales por tipo {d.senales["tipo"].value_counts().to_dict()}; {len(d.nombres_segmento)} segmentos')

## 2. Diseño: un pliegue y el registro de anclas

Arriba, un pliegue real: el bloque sintético va **delante** del train real, con fechas hábiles ficticias,
y la cola del train (contexto al predecir) sigue siendo real. El generador de ese pliegue es el ajustado
en el **ancla** τ = última fin de año válida al menos `margen_reconocimiento` sesiones antes del fin del
train, con datos ≤ τ y etiquetas solo de crisis ya reconocidas (suelo ≤ τ). Las proporciones son
**anidadas**: el bloque de 0,25 son las primeras filas del reservorio que también abren los de 0,5 y 1.
El brazo *crisis* añade solo episodios de crisis, con `p ×` los días de crisis reconocidos del train.
Debajo, el registro de anclas: un ancla vale si su ajuste tiene crisis suficientes y el generador pasa en
ella la memorización y el condicionamiento del 16; si no, se usa la válida anterior, y sin ninguna el
pliegue va sin aumento (Δ = 0 en ese pliegue).

In [ ]:
REGISTROS = {p: leer_registro(p, RES) for p in PISTAS if ruta_registro(p, RES).exists()}
guardar(fg.fig_diseno(REGISTROS, CFG), 'diseno')
display(an.resumen_anclas(REGISTROS))

## 3. Contraste principal (H1)

Por pista y detector: Δ del **exceso sobre el nulo** (score − mediana del nulo de azar persistente con la
tasa y la racha de la propia señal) entre la señal de **voto** (≥ 3 de 5 generadores; brazo completas,
p = 1, repetición 0) y el benchmark. p-valor por permutación pareada exacta de segmentos (uno por
crisis) con Holm sobre los 22 contrastes; IC 95 % por bootstrap de segmentos. «Mejora» exige las cinco
condiciones (a) Holm y Δ > 0, (b) Δ ≥ `mejora_minima`, (c) el nivel ADR-003 no empeora, (d) ningún
generador con IC por debajo de 0 y (e) Δ por encima del suelo de ruido donde lo hay. Un contraste que
aún no está cuenta con p = 1 («falta») y no encoge la familia.

In [ ]:
T1 = an.h1(DATOS, CFG)
guardar(T1, 'h1')
display(T1.reindex(columns=['pista', 'detector', 'veredicto', 'delta', 'ic_inf', 'ic_sup', 'p', 'holm', 'holm_global', 'veredicto_global', 'a_holm',
                            'b_minima', 'c_nivel', 'd_generadores', 'e_suelo', 'nivel_base', 'nivel_aumento',
                            'suelo_ruido']).round(3))
guardar(fg.fig_h1(T1, CFG), 'h1_forest')

## 4. Mapa detector × generador (exploratorio)

Δ de cada generador por separado, del voto y del control (brazo completas, p = 1). Sirve para ver si el
voto resume a los cinco o lo arrastra uno; no se elige ningún generador a posteriori.

In [ ]:
guardar(fg.fig_mapa(an.mapa_deltas(DATOS, cfg=CFG), CFG), 'mapa_detector_generador')

## 5. H2: dosis-respuesta

Δ(voto) frente a la proporción p ∈ {0; 0,25; 0,5; 1} en los dos brazos: líneas finas por detector y
gruesas para la media de cada grupo de H4, con IC bootstrap. La tabla da la pendiente MCO por detector
(Δ₀ = 0) con su IC; si H1 se cumple en un detector, se espera pendiente > 0. Secundaria, sin corrección.

In [ ]:
T2 = an.h2(DATOS, CFG)
guardar(T2, 'h2')
guardar(fg.fig_dosis(an.dosis_respuesta(DATOS, CFG), CFG), 'h2_dosis')
display(an.con_ic(T2, 'pendiente', columnas='brazo'))

## 6. H3 (control), H4 (familias) y H5 (brazos)

- **H3:** Δ(voto) − Δ(`bootstrap_regimen`), completas, p = 1. Como el voto promedia cinco ajustes y el
  control es uno, se publica también la media de los cinco Δ(generador) − Δ(control).
- **H4:** Δ medio del grupo que aprende un estado latente menos el del grupo de umbral puro (en B el grupo
  umbral queda solo en D01). Predicción: > 0. Va en la cabecera de cada fila de la figura y en la tabla.
- **H5:** Δ(crisis) − Δ(completas) con p = 1: con la misma p, «completas» añade en esperanza los mismos
  días de crisis más la calma, así que estima, con el signo cambiado, el efecto de añadir calma. Sin
  predicción de signo.

Secundarias, sin corrección: IC 95 % bootstrap pareado (azul/rojo si excluye el 0).

In [ ]:
T3, T4, T5 = an.h3(DATOS, CFG), an.h4(DATOS, CFG), an.h5(DATOS, CFG)
for t, nombre in ((T3, 'h3'), (T4, 'h4'), (T5, 'h5')):
    guardar(t, nombre)
guardar(fg.fig_contrastes(T3, T5, T4, CFG), 'h3_h4_h5')
display(T4.round(3))

## 7. Diagnósticos (exploratorios)

**Recalibración frente a aprendizaje y ocupación de estados** (primera figura). (a) Si el recall solo sube
cuando se marcan más días, el aumento recalibra (marcar siempre ya da un score crudo alto). (b) Puntos por
debajo de la diagonal: parte de la ganancia en score crudo se explica por marcar más y el exceso la
descuenta. (c) Pliegues en que un estado aparece en el bloque sintético y nunca en el train real: queda al
final del orden económico, es decir, como crisis (no se excluye; cuenta en la Δ).

**Robustez** (segunda figura). (a) Sensibilidad declarada a la fuga de diseño: Δ de H1 solo en las crisis que empiezan
después de `fin_train` del 15 + margen (la familia y los parámetros de los generadores se eligieron con
datos posteriores a los primeros pliegues). (b) Réplica de B: muestras nuevas de cada generador
(repetición 1) frente a la repetición 0. (c) Suelo de ruido: el mayor |Δ| entre semillas de los detectores
estocásticos con p = 0, frente al |Δ| de H1.

In [ ]:
guardar(fg.fig_diagnosticos(an.deltas_metricas(DATOS), an.ocupacion(FILAS), CFG), 'diagnosticos')

In [ ]:
POST, SUELO, REP = an.h1_posterior(DATOS, CFG), an.suelo_ruido(DATOS, CFG), an.replica(DATOS, CFG)
guardar(fg.fig_robustez(T1, POST, SUELO, REP), 'robustez')

## 8. Regla para el 19 y conclusiones

Regla fijada en el pre-registro: el aumento (voto, completas, p = 1) se adopta para un detector y pista
**solo** si H1 da «mejora» y H3 tiene IC > 0 en esa pista; si no, se descarta.

In [ ]:
R19 = an.regla_19(T1, T3)
guardar(R19, 'regla_19')
display(R19.round(3))

### Conclusiones

*(se redacta con los resultados)*

Siguiente: [`19_decision_final`](19_decision_final.ipynb).